# AuraGateway vLLM CUDA 12.9 offline runtime compatibility verifier v3


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.parse
import zipfile
from datetime import UTC, datetime
from pathlib import Path, PurePosixPath
from typing import Any

NOTEBOOK_NAME = "auragateway-cu129-offline-verifier-v3"
INPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_wheelhouse_v1"
OUTPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_offline_compatibility_evidence_v3"
OUTPUT_ZIP = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}.zip")
EVIDENCE_ROOT = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}")
VENV_ROOT = Path("/kaggle/working/auragateway_vllm_runtime_cu129_v3")
MAX_EXCERPT = 12000

EXPECTED_PACKAGE_COUNT = 176
EXPECTED_MANIFEST_ENTRY_COUNT = 182
EXPECTED_NON_WHEEL_ENTRY_COUNT = 6
EXPECTED_TOTAL_WHEEL_BYTES = 5727339111

EXPECTED_HASHES = {
    "requirements.in": "a120c72a5643bb65afbfe0bd3dd072f1ea89a19f57a534dd814c9bafdd41880f",
    "resolution_lock.json": (
        "1575538b0a412c9b030fc95ccada0f0527553b76f06ef6b2b72904e61c84870c"
    ),
    "materialization.lock.txt": (
        "d061bd9a7ff0a686bb462a2bd016a1f3e1aea833fbdbff353dddf96fdd623e1d"
    ),
    "requirements.lock.txt": (
        "47cb357a53ca74ca597b286768e1d0e9cb831f7431c08fad378fc42ea59b3a27"
    ),
    "install_runtime.py": (
        "68bba3ca131e9a6f36392330562985d2a644be57cf5437fd282b883741c86821"
    ),
    "runtime_manifest.json": (
        "b424d2b952d726b2f7451ebd8f48d604985f650dbe2f6d146969625618b7fc51"
    ),
    "sha256_manifest.json": (
        "789fb23ab7d9c4f28dd909e808a53a65d692c0d7b43bc44da9e974817d771b8d"
    ),
    "materialization_receipt.json": (
        "52aa42b940dd606ab5685686ab893eb085efed2a7466989f654e870f4b360589"
    ),
}

EXPECTED_RUNTIME = {
    "python": "3.12",
    "cuda_variant": "cu129",
    "vllm_binary_cuda": "12.9",
    "vllm_release": "0.19.1",
    "vllm": "0.19.1",
    "torch": "2.10.0+cu129",
    "torchaudio": "2.10.0+cu129",
    "torchvision": "0.25.0+cu129",
    "transformers": "5.5.3",
}

EXPECTED_TOP_LEVEL = frozenset(
    {
        "requirements.in",
        "resolution_lock.json",
        "materialization.lock.txt",
        "requirements.lock.txt",
        "install_runtime.py",
        "runtime_manifest.json",
        "sha256_manifest.json",
        "materialization_receipt.json",
        "wheels",
    }
)

EXPECTED_MANIFEST_CONTROL_PATHS = frozenset(
    {
        "requirements.in",
        "resolution_lock.json",
        "materialization.lock.txt",
        "requirements.lock.txt",
        "install_runtime.py",
        "runtime_manifest.json",
    }
)

REQUIRED_ROLES = (
    "base_python_runtime",
    "base_venv_import",
    "base_ensurepip_import",
    "base_ensurepip_cli",
    "venv_create_without_pip",
    "venv_python_runtime",
    "venv_ensurepip_bootstrap",
    "venv_pip_version",
    "offline_hash_locked_install",
    "pip_check",
    "gpu_topology",
    "python_runtime",
    "torch_family_runtime",
    "transformers_runtime",
    "vllm_distribution",
    "vllm_module",
    "vllm_native_extension",
)

_SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")
_MATERIALIZATION_LINE = re.compile(
    r"^(?P<name>[^ ]+) @ (?P<url>\S+) --hash=sha256:(?P<sha>[0-9a-f]{64})$"
)
_REQUIREMENTS_LINE = re.compile(
    r"^(?P<name>[^= ]+)==(?P<version>\S+) "
    r"--hash=sha256:(?P<sha>[0-9a-f]{64})$"
)


def canonical_json(payload: object) -> str:
    return json.dumps(payload, ensure_ascii=True, separators=(",", ":"), sort_keys=True)


def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def sanitize(text: str | bytes | None) -> str:
    if text is None:
        return ""
    decoded = (
        text.decode("utf-8", errors="replace") if isinstance(text, bytes) else text
    )
    bounded = decoded[-MAX_EXCERPT:]
    replacements = {
        "/kaggle/input": "<input>",
        "/kaggle/working": "<working>",
        os.environ.get("HOME", ""): "<home>",
    }
    for source, replacement in replacements.items():
        if source:
            bounded = bounded.replace(source, replacement)
    return bounded


def run_probe(role: str, argv: list[str], *, timeout: float) -> dict[str, object]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    environment = {
        **os.environ,
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "PIP_NO_INDEX": "1",
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
    }
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=environment,
        )
        return {
            "schema_version": "2.0.0",
            "command_role": role,
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": result.returncode,
            "timed_out": False,
            "status": "PASSED" if result.returncode == 0 else "FAILED",
            "stdout_excerpt": sanitize(result.stdout),
            "stderr_excerpt": sanitize(result.stderr),
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "schema_version": "2.0.0",
            "command_role": role,
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": None,
            "timed_out": True,
            "status": "FAILED",
            "stdout_excerpt": sanitize(exc.stdout),
            "stderr_excerpt": sanitize(exc.stderr),
        }


def blocked_record(role: str, *, blocked_by: str, reason: str) -> dict[str, object]:
    return {
        "schema_version": "2.0.0",
        "command_role": role,
        "status": "BLOCKED_BY_UPSTREAM_FAILURE",
        "blocked_by": blocked_by,
        "reason": reason,
        "returncode": None,
        "timed_out": False,
        "stdout_excerpt": "",
        "stderr_excerpt": "",
    }


def not_executed_record(role: str, *, reason: str) -> dict[str, object]:
    return {
        "schema_version": "2.0.0",
        "command_role": role,
        "status": "NOT_EXECUTED",
        "reason": reason,
        "returncode": None,
        "timed_out": False,
        "stdout_excerpt": "",
        "stderr_excerpt": "",
    }


def block_remaining(
    records: list[dict[str, object]],
    *,
    blocked_by: str,
    reason: str,
) -> None:
    observed = {str(record["command_role"]) for record in records}
    for role in REQUIRED_ROLES:
        if role not in observed:
            records.append(blocked_record(role, blocked_by=blocked_by, reason=reason))


def summarize_role_states(
    records: list[dict[str, object]],
) -> dict[str, list[str]]:
    observed = {str(record["command_role"]): record for record in records}
    for role in REQUIRED_ROLES:
        if role not in observed:
            record = not_executed_record(role, reason="role record was not emitted")
            records.append(record)
            observed[role] = record
    return {
        "failed_required_roles": sorted(
            role for role in REQUIRED_ROLES if observed[role]["status"] == "FAILED"
        ),
        "blocked_required_roles": sorted(
            role
            for role in REQUIRED_ROLES
            if observed[role]["status"] == "BLOCKED_BY_UPSTREAM_FAILURE"
        ),
        "not_executed_required_roles": sorted(
            role
            for role in REQUIRED_ROLES
            if observed[role]["status"] == "NOT_EXECUTED"
        ),
    }


def require_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError(f"expected one JSON object: {path.name}")
    return payload


def validate_control_hashes(input_root: Path) -> None:
    for relative, expected in EXPECTED_HASHES.items():
        path = input_root / relative
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f"control file is missing or unsafe: {relative}")
        if streaming_sha256(path) != expected:
            raise RuntimeError(f"control file identity drifted: {relative}")


def parse_manifest_entries(payload: dict[str, Any]) -> list[dict[str, Any]]:
    if payload.get("schema_version") != "1.0.0":
        raise RuntimeError("wheelhouse SHA-256 manifest schema drifted")
    entries = payload.get("entries")
    if not isinstance(entries, list) or len(entries) != EXPECTED_MANIFEST_ENTRY_COUNT:
        raise RuntimeError("wheelhouse SHA-256 manifest entry count drifted")
    parsed: list[dict[str, Any]] = []
    for entry in entries:
        if not isinstance(entry, dict):
            raise RuntimeError("wheelhouse SHA-256 entry is invalid")
        path = entry.get("path")
        digest = entry.get("sha256")
        size_bytes = entry.get("size_bytes")
        if (
            not isinstance(path, str)
            or not isinstance(digest, str)
            or _SHA256_PATTERN.fullmatch(digest) is None
            or not isinstance(size_bytes, int)
            or size_bytes < 0
        ):
            raise RuntimeError("wheelhouse SHA-256 entry fields are invalid")
        relative = PurePosixPath(path)
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("wheelhouse SHA-256 path is unsafe")
        parsed.append({"path": path, "sha256": digest, "size_bytes": size_bytes})
    paths = [str(entry["path"]) for entry in parsed]
    hashes = [str(entry["sha256"]) for entry in parsed]
    if len(paths) != len(set(paths)):
        raise RuntimeError("wheelhouse SHA-256 paths are not unique")
    if len(hashes) != len(set(hashes)):
        raise RuntimeError("wheelhouse SHA-256 identities are not unique")
    return parsed


def validate_manifest_files(input_root: Path, entries: list[dict[str, Any]]) -> None:
    for entry in entries:
        relative = PurePosixPath(str(entry["path"]))
        path = input_root.joinpath(*relative.parts)
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(
                f"wheelhouse payload member is missing or unsafe: {relative}"
            )
        if path.stat().st_size != entry["size_bytes"]:
            raise RuntimeError(f"wheelhouse payload size drifted: {relative}")
        if streaming_sha256(path) != entry["sha256"]:
            raise RuntimeError(f"wheelhouse payload identity drifted: {relative}")


def parse_materialization_lock(path: Path) -> dict[str, dict[str, str]]:
    lines = [line.strip() for line in path.read_text(encoding="utf-8").splitlines()]
    lines = [line for line in lines if line]
    if len(lines) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError("materialization lock package count drifted")
    parsed: dict[str, dict[str, str]] = {}
    for line in lines:
        match = _MATERIALIZATION_LINE.fullmatch(line)
        if match is None:
            raise RuntimeError("materialization lock line is invalid")
        record = match.groupdict()
        key = normalize_name(record["name"])
        if key in parsed:
            raise RuntimeError("materialization lock package names are not unique")
        parsed[key] = record
    return parsed


def parse_requirements_lock(path: Path) -> dict[str, dict[str, str]]:
    lines = [line.strip() for line in path.read_text(encoding="utf-8").splitlines()]
    lines = [line for line in lines if line]
    if len(lines) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError("requirements lock package count drifted")
    parsed: dict[str, dict[str, str]] = {}
    for line in lines:
        match = _REQUIREMENTS_LINE.fullmatch(line)
        if match is None:
            raise RuntimeError("requirements lock line is invalid")
        record = match.groupdict()
        key = normalize_name(record["name"])
        if key in parsed:
            raise RuntimeError("requirements lock package names are not unique")
        parsed[key] = record
    return parsed


def validate_resolution_closure(
    input_root: Path,
    entries: list[dict[str, Any]],
) -> dict[str, object]:
    resolution = require_json_object(input_root / "resolution_lock.json")
    if (
        resolution.get("schema_version") != "1.0.0"
        or resolution.get("package_count") != EXPECTED_PACKAGE_COUNT
        or resolution.get("host_count") != 5
        or resolution.get("wildcard_domains_permitted") is not False
        or resolution.get("review_decision") != "APPROVED_AS_EXACT_LOCKED_CLOSURE"
    ):
        raise RuntimeError("resolution lock contract drifted")
    records = resolution.get("records")
    if not isinstance(records, list) or len(records) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError("resolution lock records drifted")

    entry_by_path = {str(entry["path"]): entry for entry in entries}
    wheel_entries = {
        path: entry
        for path, entry in entry_by_path.items()
        if path.startswith("wheels/")
    }
    control_paths = set(entry_by_path) - set(wheel_entries)
    if len(wheel_entries) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError("wheel manifest package count drifted")
    if control_paths != EXPECTED_MANIFEST_CONTROL_PATHS:
        raise RuntimeError("wheelhouse control manifest topology drifted")

    materialization_lock = parse_materialization_lock(
        input_root / "materialization.lock.txt"
    )
    requirements_lock = parse_requirements_lock(input_root / "requirements.lock.txt")

    observed_names: set[str] = set()
    observed_wheel_paths: set[str] = set()
    for record in records:
        if not isinstance(record, dict):
            raise RuntimeError("resolution lock record is invalid")
        name = record.get("normalized_name")
        version = record.get("version")
        artifact_filename = record.get("artifact_filename")
        artifact_sha256 = record.get("sha256")
        sanitized_url = record.get("sanitized_url")
        if not all(
            isinstance(value, str)
            for value in (
                name,
                version,
                artifact_filename,
                artifact_sha256,
                sanitized_url,
            )
        ):
            raise RuntimeError("resolution lock record fields are invalid")
        key = normalize_name(name)
        if key in observed_names:
            raise RuntimeError("resolution lock package names are not unique")
        observed_names.add(key)

        decoded_filename = urllib.parse.unquote(artifact_filename)
        wheel_path = f"wheels/{decoded_filename}"
        observed_wheel_paths.add(wheel_path)
        manifest_entry = wheel_entries.get(wheel_path)
        if manifest_entry is None or manifest_entry["sha256"] != artifact_sha256:
            raise RuntimeError(f"resolution-to-wheel identity drifted: {key}")

        materialized = materialization_lock.get(key)
        if (
            materialized is None
            or materialized["url"] != sanitized_url
            or materialized["sha"] != artifact_sha256
        ):
            raise RuntimeError(f"materialization lock drifted: {key}")

        requirement = requirements_lock.get(key)
        if (
            requirement is None
            or requirement["version"] != version
            or requirement["sha"] != artifact_sha256
        ):
            raise RuntimeError(f"requirements lock drifted: {key}")

    if observed_wheel_paths != set(wheel_entries):
        raise RuntimeError("wheel manifest contains unexpected package artifacts")

    total_wheel_bytes = sum(
        int(entry["size_bytes"]) for entry in wheel_entries.values()
    )
    if total_wheel_bytes != EXPECTED_TOTAL_WHEEL_BYTES:
        raise RuntimeError("wheelhouse byte count drifted")

    return {
        "manifest_entry_count": len(entries),
        "wheel_entry_count": len(wheel_entries),
        "non_wheel_entry_count": len(control_paths),
        "total_wheel_bytes": total_wheel_bytes,
        "package_count": len(observed_names),
    }


def validate_receipt(input_root: Path) -> None:
    receipt = require_json_object(input_root / "materialization_receipt.json")
    expected = {
        "schema_version": "1.2.0",
        "notebook_name": "auragateway-cu129-wheelhouse-materializer-v1",
        "output_directory": INPUT_DIRECTORY_NAME,
        "materialization_status": "PASSED",
        "package_count": EXPECTED_PACKAGE_COUNT,
        "total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
        "resolution_lock_sha256": EXPECTED_HASHES["resolution_lock.json"],
        "materialization_lock_sha256": EXPECTED_HASHES["materialization.lock.txt"],
        "requirements_lock_sha256": EXPECTED_HASHES["requirements.lock.txt"],
        "runtime_manifest_sha256": EXPECTED_HASHES["runtime_manifest.json"],
        "sha256_manifest_sha256": EXPECTED_HASHES["sha256_manifest.json"],
        "pip_download_subcommand_performed": True,
        "pip_resolution_artifact_transfer_observed": True,
        "pip_resolution_transfer_event_count": 358,
        "package_installation_performed": False,
        "model_requests_performed": 0,
        "qualification_claimed": False,
        "credentials_used": False,
        "customer_data_used": False,
    }
    drift = sorted(key for key, value in expected.items() if receipt.get(key) != value)
    if drift:
        raise RuntimeError("materialization receipt drifted: " + ", ".join(drift))


def validate_runtime_manifest(input_root: Path) -> None:
    runtime = require_json_object(input_root / "runtime_manifest.json")
    for key, value in EXPECTED_RUNTIME.items():
        if runtime.get(key) != value:
            raise RuntimeError(f"runtime manifest drifted at {key}")
    if (
        runtime.get("schema_version") != "1.2.0"
        or runtime.get("package_count") != EXPECTED_PACKAGE_COUNT
        or runtime.get("network_required_for_installation") is not False
        or runtime.get("model_requests_performed") != 0
        or runtime.get("qualification_claimed") is not False
    ):
        raise RuntimeError("runtime manifest safety contract drifted")


def validate_input(input_root: Path) -> dict[str, object]:
    if not input_root.is_dir() or input_root.is_symlink():
        raise RuntimeError("governed wheelhouse root is missing or unsafe")
    observed_topology = {path.name for path in input_root.iterdir()}
    if observed_topology != EXPECTED_TOP_LEVEL:
        raise RuntimeError("wheelhouse top-level topology drifted")

    validate_control_hashes(input_root)
    installer_source = (input_root / "install_runtime.py").read_text(encoding="utf-8")
    required_install_flags = ('"--no-index"', '"--find-links"', '"--require-hashes"')
    if any(flag not in installer_source for flag in required_install_flags):
        raise RuntimeError("offline installer lost a required isolation flag")

    sha_manifest = require_json_object(input_root / "sha256_manifest.json")
    entries = parse_manifest_entries(sha_manifest)
    validate_manifest_files(input_root, entries)
    closure = validate_resolution_closure(input_root, entries)
    validate_receipt(input_root)
    validate_runtime_manifest(input_root)

    free_bytes = shutil.disk_usage("/kaggle/working").free
    if free_bytes < 12 * 1024**3:
        raise RuntimeError(
            "insufficient writable disk for isolated runtime installation"
        )

    return {
        "schema_version": "2.0.0",
        "status": "PASSED",
        "input_directory_name": INPUT_DIRECTORY_NAME,
        "input_sha256_manifest_sha256": EXPECTED_HASHES["sha256_manifest.json"],
        "input_materialization_receipt_sha256": EXPECTED_HASHES[
            "materialization_receipt.json"
        ],
        "input_runtime_manifest_sha256": EXPECTED_HASHES["runtime_manifest.json"],
        "resolution_lock_sha256": EXPECTED_HASHES["resolution_lock.json"],
        "package_count": closure["package_count"],
        "manifest_entry_count": closure["manifest_entry_count"],
        "wheel_entry_count": closure["wheel_entry_count"],
        "non_wheel_entry_count": closure["non_wheel_entry_count"],
        "total_wheel_bytes": closure["total_wheel_bytes"],
        "working_free_bytes_before_install": free_bytes,
        "installer_contract_validated": True,
        "install_runtime_script_executed": False,
        "network_access_requested": False,
        "credentials_used": False,
        "customer_data_used": False,
        "model_requests_performed": 0,
        "qualification_claimed": False,
    }


def reject_semantic(
    observed: dict[str, dict[str, object]],
    role: str,
    message: str,
) -> None:
    record = observed.get(role)
    if record is None or record["status"] != "PASSED":
        return
    record["status"] = "FAILED"
    record["semantic_error"] = message


def apply_semantic_checks(records: list[dict[str, object]]) -> None:
    observed = {str(record["command_role"]): record for record in records}

    for role in ("base_python_runtime", "venv_python_runtime", "python_runtime"):
        record = observed.get(role)
        if (
            record is not None
            and record["status"] == "PASSED"
            and not str(record["stdout_excerpt"]).strip().startswith("3.12.")
        ):
            reject_semantic(observed, role, "Python runtime is not 3.12.x")

    for role in ("base_venv_import",):
        record = observed.get(role)
        if record is not None and str(record["stdout_excerpt"]).strip() != "ok":
            reject_semantic(observed, role, "venv import did not emit ok")

    ensurepip_import = observed.get("base_ensurepip_import")
    if ensurepip_import is not None and ensurepip_import["status"] == "PASSED":
        try:
            payload = json.loads(str(ensurepip_import["stdout_excerpt"]).strip())
        except json.JSONDecodeError:
            reject_semantic(
                observed,
                "base_ensurepip_import",
                "ensurepip import output is not JSON",
            )
        else:
            if payload.get("module") != "ensurepip" or not isinstance(
                payload.get("version"), str
            ):
                reject_semantic(
                    observed,
                    "base_ensurepip_import",
                    "ensurepip import identity drifted",
                )

    ensurepip_cli = observed.get("base_ensurepip_cli")
    if (
        ensurepip_cli is not None
        and ensurepip_cli["status"] == "PASSED"
        and "pip" not in str(ensurepip_cli["stdout_excerpt"]).lower()
    ):
        reject_semantic(observed, "base_ensurepip_cli", "ensurepip CLI output drifted")

    pip_version = observed.get("venv_pip_version")
    if (
        pip_version is not None
        and pip_version["status"] == "PASSED"
        and not str(pip_version["stdout_excerpt"]).strip().startswith("pip ")
    ):
        reject_semantic(observed, "venv_pip_version", "venv pip identity drifted")

    topology = observed.get("gpu_topology")
    if topology is not None and topology["status"] == "PASSED":
        lines = [
            line.strip()
            for line in str(topology["stdout_excerpt"]).splitlines()
            if line.strip()
        ]
        if len(lines) != 2:
            reject_semantic(
                observed, "gpu_topology", "expected exactly two GPU records"
            )
        else:
            for index, line in enumerate(lines):
                parts = [part.strip() for part in line.split(",")]
                if len(parts) != 5 or parts[0] != str(index):
                    reject_semantic(
                        observed,
                        "gpu_topology",
                        "GPU topology record is malformed",
                    )
                    break
                if parts[1] not in {"Tesla T4", "NVIDIA T4"} or parts[3] != "7.5":
                    reject_semantic(
                        observed,
                        "gpu_topology",
                        "GPU topology is not two T4 devices",
                    )
                    break

    torch_runtime = observed.get("torch_family_runtime")
    if torch_runtime is not None and torch_runtime["status"] == "PASSED":
        try:
            payload = json.loads(str(torch_runtime["stdout_excerpt"]).strip())
        except json.JSONDecodeError:
            reject_semantic(
                observed,
                "torch_family_runtime",
                "Torch-family runtime output is not JSON",
            )
        else:
            expected = {
                "torch": "2.10.0+cu129",
                "torchaudio": "2.10.0+cu129",
                "torchvision": "0.25.0+cu129",
                "cuda": "12.9",
                "available": True,
                "device_count": 2,
            }
            if payload != expected:
                reject_semantic(
                    observed,
                    "torch_family_runtime",
                    "Torch-family or CUDA runtime identity drifted",
                )

    expected_stdout = {
        "transformers_runtime": "5.5.3",
        "vllm_distribution": "0.19.1",
        "vllm_module": "0.19.1",
        "vllm_native_extension": "ok",
    }
    for role, expected_value in expected_stdout.items():
        record = observed.get(role)
        if (
            record is not None
            and record["status"] == "PASSED"
            and str(record["stdout_excerpt"]).strip() != expected_value
        ):
            reject_semantic(observed, role, f"expected {expected_value}")


def append_probe(
    records: list[dict[str, object]],
    role: str,
    argv: list[str],
    *,
    timeout: float,
) -> bool:
    record = run_probe(role, argv, timeout=timeout)
    records.append(record)
    return record["status"] == "PASSED"


def write_evidence_record(name: str, payload: object) -> None:
    (EVIDENCE_ROOT / name).write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
    )


def main() -> int:
    if os.environ.get("AURAGATEWAY_CUSTOMER_DATA_PRESENT") == "1":
        raise RuntimeError("customer data is prohibited")

    if EVIDENCE_ROOT.exists():
        shutil.rmtree(EVIDENCE_ROOT)
    EVIDENCE_ROOT.mkdir(parents=True)
    if VENV_ROOT.exists():
        shutil.rmtree(VENV_ROOT)
    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()

    records: list[dict[str, object]] = []
    input_validation: dict[str, object]
    input_root: Path | None = None
    matches = tuple(Path("/kaggle/input").rglob(INPUT_DIRECTORY_NAME))

    try:
        if len(matches) != 1:
            raise RuntimeError("expected exactly one governed wheelhouse directory")
        input_root = matches[0]
        input_validation = validate_input(input_root)
    except Exception as exc:
        input_validation = {
            "schema_version": "2.0.0",
            "status": "FAILED",
            "error_type": type(exc).__name__,
            "error_message": sanitize(str(exc)),
            "network_access_requested": False,
            "credentials_used": False,
            "customer_data_used": False,
            "model_requests_performed": 0,
            "qualification_claimed": False,
        }
        block_remaining(
            records,
            blocked_by="input_validation",
            reason="governed input validation failed",
        )

    if input_validation["status"] == "PASSED" and input_root is not None:
        base_probes = (
            (
                "base_python_runtime",
                [
                    sys.executable,
                    "-c",
                    "import platform; print(platform.python_version())",
                ],
                30.0,
            ),
            (
                "base_venv_import",
                [sys.executable, "-c", "import venv; print('ok')"],
                30.0,
            ),
            (
                "base_ensurepip_import",
                [
                    sys.executable,
                    "-c",
                    (
                        "import ensurepip,json;"
                        "print(json.dumps({'module':'ensurepip',"
                        "'version':ensurepip.version()}))"
                    ),
                ],
                30.0,
            ),
            (
                "base_ensurepip_cli",
                [sys.executable, "-m", "ensurepip", "--version"],
                60.0,
            ),
        )
        for role, argv, timeout in base_probes:
            if not append_probe(records, role, argv, timeout=timeout):
                block_remaining(
                    records,
                    blocked_by=role,
                    reason="base interpreter bootstrap prerequisite failed",
                )
                break

    observed = {str(record["command_role"]): record for record in records}
    if (
        input_validation["status"] == "PASSED"
        and input_root is not None
        and not any(record["status"] == "FAILED" for record in records)
    ):
        role = "venv_create_without_pip"
        if not append_probe(
            records,
            role,
            [sys.executable, "-m", "venv", "--without-pip", str(VENV_ROOT)],
            timeout=180.0,
        ):
            block_remaining(
                records,
                blocked_by=role,
                reason="virtual environment creation without pip failed",
            )

    python = VENV_ROOT / "bin" / "python"
    observed = {str(record["command_role"]): record for record in records}
    if (
        "venv_create_without_pip" in observed
        and observed["venv_create_without_pip"]["status"] == "PASSED"
    ):
        role = "venv_python_runtime"
        if not append_probe(
            records,
            role,
            [str(python), "-c", "import platform; print(platform.python_version())"],
            timeout=30.0,
        ):
            block_remaining(
                records,
                blocked_by=role,
                reason="new virtual environment interpreter failed",
            )

    observed = {str(record["command_role"]): record for record in records}
    if (
        "venv_python_runtime" in observed
        and observed["venv_python_runtime"]["status"] == "PASSED"
    ):
        role = "venv_ensurepip_bootstrap"
        if not append_probe(
            records,
            role,
            [str(python), "-m", "ensurepip", "--upgrade", "--default-pip"],
            timeout=300.0,
        ):
            block_remaining(
                records,
                blocked_by=role,
                reason="captured ensurepip bootstrap failed",
            )

    observed = {str(record["command_role"]): record for record in records}
    if (
        "venv_ensurepip_bootstrap" in observed
        and observed["venv_ensurepip_bootstrap"]["status"] == "PASSED"
    ):
        role = "venv_pip_version"
        if not append_probe(
            records,
            role,
            [str(python), "-m", "pip", "--version"],
            timeout=60.0,
        ):
            block_remaining(
                records,
                blocked_by=role,
                reason="bootstrapped pip is not executable",
            )

    observed = {str(record["command_role"]): record for record in records}
    if (
        "venv_pip_version" in observed
        and observed["venv_pip_version"]["status"] == "PASSED"
        and input_root is not None
    ):
        role = "offline_hash_locked_install"
        if not append_probe(
            records,
            role,
            [
                str(python),
                "-m",
                "pip",
                "install",
                "--no-index",
                "--find-links",
                str(input_root / "wheels"),
                "--require-hashes",
                "-r",
                str(input_root / "requirements.lock.txt"),
            ],
            timeout=2400.0,
        ):
            block_remaining(
                records,
                blocked_by=role,
                reason="hash-locked offline package installation failed",
            )

    observed = {str(record["command_role"]): record for record in records}
    if (
        "offline_hash_locked_install" in observed
        and observed["offline_hash_locked_install"]["status"] == "PASSED"
    ):
        runtime_probes = (
            (
                "pip_check",
                [str(python), "-m", "pip", "check"],
                120.0,
            ),
            (
                "gpu_topology",
                [
                    "nvidia-smi",
                    "--query-gpu=index,name,memory.total,compute_cap,driver_version",
                    "--format=csv,noheader,nounits",
                ],
                30.0,
            ),
            (
                "python_runtime",
                [
                    str(python),
                    "-c",
                    "import platform; print(platform.python_version())",
                ],
                30.0,
            ),
            (
                "torch_family_runtime",
                [
                    str(python),
                    "-c",
                    (
                        "import json,torch,torchaudio,torchvision;"
                        "print(json.dumps({"
                        "'torch':torch.__version__,"
                        "'torchaudio':torchaudio.__version__,"
                        "'torchvision':torchvision.__version__,"
                        "'cuda':torch.version.cuda,"
                        "'available':torch.cuda.is_available(),"
                        "'device_count':torch.cuda.device_count()}))"
                    ),
                ],
                180.0,
            ),
            (
                "transformers_runtime",
                [
                    str(python),
                    "-c",
                    "import transformers; print(transformers.__version__)",
                ],
                60.0,
            ),
            (
                "vllm_distribution",
                [
                    str(python),
                    "-c",
                    (
                        "import importlib.metadata;"
                        "print(importlib.metadata.version('vllm'))"
                    ),
                ],
                60.0,
            ),
            (
                "vllm_module",
                [str(python), "-c", "import vllm; print(vllm.__version__)"],
                180.0,
            ),
            (
                "vllm_native_extension",
                [
                    str(python),
                    "-c",
                    ("import importlib;importlib.import_module('vllm._C');print('ok')"),
                ],
                180.0,
            ),
        )
        for role, argv, timeout in runtime_probes:
            records.append(run_probe(role, argv, timeout=timeout))

    apply_semantic_checks(records)
    states = summarize_role_states(records)
    observed = {str(record["command_role"]): record for record in records}
    first_divergence = next(
        (role for role in REQUIRED_ROLES if observed[role]["status"] == "FAILED"),
        None,
    )
    install_record = observed["offline_hash_locked_install"]
    package_installation_started = install_record["status"] in {"PASSED", "FAILED"}

    write_evidence_record("00_input_validation.json", input_validation)
    for index, role in enumerate(REQUIRED_ROLES, start=1):
        write_evidence_record(f"10_{index:02d}_{role}.json", observed[role])

    passed = (
        input_validation["status"] == "PASSED"
        and not states["failed_required_roles"]
        and not states["blocked_required_roles"]
        and not states["not_executed_required_roles"]
    )
    summary = {
        "schema_version": "3.0.0",
        "diagnostic_id": NOTEBOOK_NAME,
        "captured_at": datetime.now(UTC).isoformat(timespec="seconds"),
        "status": "PASSED" if passed else "FAILED",
        "first_divergence": first_divergence,
        **states,
        "input_sha256_manifest_sha256": EXPECTED_HASHES["sha256_manifest.json"],
        "input_materialization_receipt_sha256": EXPECTED_HASHES[
            "materialization_receipt.json"
        ],
        "input_runtime_manifest_sha256": EXPECTED_HASHES["runtime_manifest.json"],
        "resolution_lock_sha256": EXPECTED_HASHES["resolution_lock.json"],
        "package_count": EXPECTED_PACKAGE_COUNT,
        "package_installation_started": package_installation_started,
        "install_runtime_script_executed": False,
        "model_requests_performed": 0,
        "benchmark_trajectory_requests_performed": 0,
        "qualification_claimed": False,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
    }
    write_evidence_record("90_summary.json", summary)

    payload_paths = tuple(sorted(EVIDENCE_ROOT.iterdir(), key=lambda path: path.name))
    sha_entries = [
        {
            "path": path.name,
            "sha256": streaming_sha256(path),
            "size_bytes": path.stat().st_size,
        }
        for path in payload_paths
    ]
    write_evidence_record(
        "99_evidence_sha256.json",
        {"schema_version": "1.0.0", "entries": sha_entries},
    )

    with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(EVIDENCE_ROOT.iterdir(), key=lambda item: item.name):
            archive.write(path, arcname=path.name)

    print(f"artifact={OUTPUT_ZIP}")
    print(f"size_bytes={OUTPUT_ZIP.stat().st_size}")
    print(f"sha256={streaming_sha256(OUTPUT_ZIP)}")
    print(f"offline_compatibility_status={summary['status']}")
    print(f"first_divergence={first_divergence}")
    print("failed_required_roles=" + canonical_json(states["failed_required_roles"]))
    print("blocked_required_roles=" + canonical_json(states["blocked_required_roles"]))
    print(
        "not_executed_required_roles="
        + canonical_json(states["not_executed_required_roles"])
    )
    print("model_requests_performed=0")
    print("qualification_claimed=false")
    print("upload_only_this_file=true")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
